# Module 03 — Lab: Tool use

You will build:
1. A tool loop with two real tools (weather + calculator).
2. A structured invoice extractor using forced tool choice.
3. A parallel-call dispatcher.

In [ ]:
import os, ast, math, json
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv('../.env')
client = Anthropic()
MODEL = os.getenv('ANTHROPIC_MODEL', 'claude-sonnet-4-6')

## 1. Tool definitions

In [ ]:
TOOLS = [
    {
        'name': 'get_weather',
        'description': 'Get current weather for a city. Returns a short summary string.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'city': {'type': 'string', 'description': 'City name only.'},
            },
            'required': ['city'],
        },
    },
    {
        'name': 'calculator',
        'description': 'Evaluate an arithmetic expression. Supports +, -, *, /, **, sqrt(), and parentheses.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'expression': {'type': 'string'},
            },
            'required': ['expression'],
        },
    },
]

# --- tool implementations (mocked / safe) ---
_WEATHER_MOCK = {
    'tokyo':   '17C, light rain',
    'london':  '12C, cloudy',
    'lagos':   '30C, humid',
    'pune':    '28C, sunny',
}

def get_weather(city: str) -> str:
    return _WEATHER_MOCK.get(city.strip().lower(), 'unknown city')

_ALLOWED = {'sqrt': math.sqrt}

def calculator(expression: str) -> str:
    # Very narrow safe evaluator: AST walk, numeric ops only.
    tree = ast.parse(expression, mode='eval')
    def walk(node):
        if isinstance(node, ast.Expression): return walk(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value,(int,float)): return node.value
        if isinstance(node, ast.BinOp):
            l, r = walk(node.left), walk(node.right)
            return {ast.Add: l+r, ast.Sub: l-r, ast.Mult: l*r, ast.Div: l/r, ast.Pow: l**r}[type(node.op)]
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub): return -walk(node.operand)
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in _ALLOWED:
            return _ALLOWED[node.func.id](*(walk(a) for a in node.args))
        raise ValueError(f'not allowed: {ast.dump(node)}')
    return str(walk(tree))

TOOL_FNS = {'get_weather': get_weather, 'calculator': calculator}
print(calculator('17 * 23 + sqrt(144)'))

## 2. The tool loop

In [ ]:
def run_with_tools(user_text, max_turns=8, verbose=True):
    messages = [{'role':'user','content':user_text}]
    for turn in range(max_turns):
        r = client.messages.create(
            model=MODEL, max_tokens=1024,
            tools=TOOLS, messages=messages,
        )
        messages.append({'role':'assistant','content': r.content})
        if verbose:
            print(f'\n--- turn {turn}  stop={r.stop_reason} ---')
            for b in r.content:
                if b.type == 'text':       print('TEXT:', b.text)
                elif b.type == 'tool_use': print(f'CALL {b.name}({b.input})')
        if r.stop_reason == 'end_turn':
            return ''.join(b.text for b in r.content if b.type == 'text')
        if r.stop_reason == 'tool_use':
            results = []
            for b in r.content:
                if b.type != 'tool_use': continue
                try:
                    out = TOOL_FNS[b.name](**b.input)
                    results.append({'type':'tool_result','tool_use_id': b.id,'content': str(out)})
                except Exception as e:
                    results.append({'type':'tool_result','tool_use_id': b.id,
                                     'content': f'ERROR: {e}', 'is_error': True})
            messages.append({'role':'user','content': results})
            continue
        raise RuntimeError(f'unexpected stop_reason: {r.stop_reason}')
    raise RuntimeError('max_turns exceeded')

print(run_with_tools(
    "What's the weather in Tokyo? Also what is 17 * 23 + sqrt(144)?"
))

## 3. Structured output via forced tool

In [ ]:
EXTRACTOR = {
    'name': 'save_invoice',
    'description': 'Record an invoice extracted from the document.',
    'input_schema': {
        'type': 'object',
        'properties': {
            'vendor': {'type': 'string'},
            'total': {'type': 'number'},
            'currency': {'type': 'string', 'enum': ['USD','EUR','INR','GBP']},
            'due_date': {'type': 'string', 'description': 'YYYY-MM-DD'},
        },
        'required': ['vendor','total','currency','due_date'],
    },
}

INVOICE = '''Invoice from Acme Robotics Pvt Ltd.
Amount due: INR 84,500. Payment due by 30 June 2026. Thank you!'''

r = client.messages.create(
    model=MODEL, max_tokens=512,
    tools=[EXTRACTOR],
    tool_choice={'type':'tool','name':'save_invoice'},
    messages=[{'role':'user','content': INVOICE}],
)
extracted = next(b.input for b in r.content if b.type == 'tool_use')
print(json.dumps(extracted, indent=2))

## 4. Parallel weather lookups

In [ ]:
import time
messages = [{'role':'user','content':"What's the weather in Tokyo, London, and Lagos? List each."}]
r = client.messages.create(model=MODEL, max_tokens=512, tools=TOOLS, messages=messages)
calls = [b for b in r.content if b.type == 'tool_use']
print(f'parallel tool calls: {len(calls)}')
for c in calls:
    print(' ', c.name, c.input)

# In production you'd asyncio.gather these. With mocked sync fns we just loop.
t0 = time.perf_counter()
results = []
for c in calls:
    results.append({'type':'tool_result','tool_use_id': c.id,
                     'content': TOOL_FNS[c.name](**c.input)})
messages.append({'role':'assistant','content': r.content})
messages.append({'role':'user','content': results})

final = client.messages.create(model=MODEL, max_tokens=512, tools=TOOLS, messages=messages)
print(f'\n--- final ({(time.perf_counter()-t0)*1000:.0f} ms) ---')
print(final.content[0].text)